# Aula 13 — Backprop vetorizado em uma MLP

Este laboratório implementa uma rede de duas camadas para classificação multiclasse usando **NumPy puro**, sem autograd. Cada linha é um exemplo. O objetivo é auditar a álgebra do lote: shapes, redução média, gradientes dos biases e equivalência com o cálculo individual.

**Dependências:** Python ≥ 3.11, NumPy ≥ 1.26, Matplotlib ≥ 3.8 e nbformat ≥ 5.9 para validar o arquivo.  
**Seed:** `20260913`.  
**Dados:** exemplo manual e dados sintéticos gerados localmente; não há download, credenciais ou alegação de generalização.

In [ ]:
import platform
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260913
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)

## 1. Convenção e exemplo manual

Usaremos (X\in\mathbb{R}^{m\times d}), (W_1\in\mathbb{R}^{d\times h}), (b_1\in\mathbb{R}^{h}), (W_2\in\mathbb{R}^{h\times K}) e (b_2\in\mathbb{R}^{K}). O NumPy transmite cada bias pelas linhas do lote.

In [ ]:
X_small = np.array([[1.0, 2.0], [-1.0, 1.0]])
Y_small = np.eye(2)[[0, 1]]
params_small = {
    "W1": np.array([[0.5, -0.2], [0.3, 0.4]]),
    "b1": np.array([0.1, -0.1]),
    "W2": np.array([[0.7, -0.5], [-0.3, 0.8]]),
    "b2": np.array([0.05, -0.05]),
}

assert X_small.shape == (2, 2)
assert Y_small.shape == (2, 2)
assert np.all(Y_small.sum(axis=1) == 1)
print("X =\n", X_small)
print("Y =\n", Y_small)

## 2. Forward estável e cache explícito

A loss usa a média dos exemplos. Calculamos log-probabilidades por log-sum-exp, evitando `log(softmax)` instável. O cache contém cópias dos arrays necessários para que uma atualização posterior dos parâmetros não altere o passado.

In [ ]:
def validate_inputs(X, Y, params):
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    W1, b1, W2, b2 = (np.asarray(params[k], dtype=float) for k in ("W1", "b1", "W2", "b2"))
    assert X.ndim == Y.ndim == W1.ndim == W2.ndim == 2
    assert b1.ndim == b2.ndim == 1
    m, d = X.shape
    assert m > 0 and Y.shape[0] == m
    assert W1.shape[0] == d and W1.shape[1] == b1.shape[0]
    assert W2.shape[0] == b1.shape[0] and W2.shape[1] == b2.shape[0]
    assert Y.shape[1] == b2.shape[0]
    assert np.allclose(Y.sum(axis=1), 1.0) and np.all(Y >= 0)
    assert all(np.all(np.isfinite(a)) for a in (X, Y, W1, b1, W2, b2))
    return X, Y, W1, b1, W2, b2


def forward(X, Y, params):
    X, Y, W1, b1, W2, b2 = validate_inputs(X, Y, params)
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    Z2 = A1 @ W2 + b2
    row_max = Z2.max(axis=1, keepdims=True)
    shifted = Z2 - row_max
    logsumexp = row_max + np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    log_probs = Z2 - logsumexp
    P = np.exp(log_probs)
    loss = -np.sum(Y * log_probs) / X.shape[0]
    cache = {name: value.copy() for name, value in {
        "X": X, "Y": Y, "W1": W1, "b1": b1, "Z1": Z1,
        "A1": A1, "W2": W2, "b2": b2, "Z2": Z2, "P": P,
    }.items()}
    assert np.isscalar(loss) and np.isfinite(loss)
    assert np.allclose(P.sum(axis=1), 1.0)
    return float(loss), P, cache


loss_small, P_small, cache_small = forward(X_small, Y_small, params_small)
print("Z1 =\n", cache_small["Z1"])
print("A1 =\n", cache_small["A1"])
print("Z2 =\n", cache_small["Z2"])
print("P =\n", P_small)
print(f"Cross-entropy média = {loss_small:.9f}")

## 3. Backward vetorizado

Para softmax com cross-entropy média, (G_2=(P-Y)/m). As multiplicações (A_1^\top G_2) e (X^\top G_1) contraem o eixo dos exemplos; os biases acumulam com soma no eixo 0.

In [ ]:
def backward(cache):
    X, Y = cache["X"], cache["Y"]
    W1, A1, W2, P = cache["W1"], cache["A1"], cache["W2"], cache["P"]
    m = X.shape[0]

    G2 = (P - Y) / m
    dW2 = A1.T @ G2
    db2 = G2.sum(axis=0)
    dA1 = G2 @ W2.T
    G1 = dA1 * (1.0 - A1**2)
    dW1 = X.T @ G1
    db1 = G1.sum(axis=0)
    dX = G1 @ W1.T

    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    for name, grad in grads.items():
        assert grad.shape == cache[name].shape, (name, grad.shape, cache[name].shape)
        assert np.all(np.isfinite(grad))
    assert dX.shape == X.shape and np.all(np.isfinite(dX))
    assert np.allclose(G2.sum(axis=1), 0.0, atol=1e-12)
    aux = {"G2": G2, "dA1": dA1, "G1": G1}
    return grads, dX, aux


grads_small, dX_small, aux_small = backward(cache_small)
for name, grad in grads_small.items():
    print(f"d{name} =\n{grad}")
print("dX =\n", dX_small)

### Auditoria dos shapes

Os gradientes de parâmetros devem ter os mesmos shapes dos parâmetros; (dX) deve ter o shape da entrada. Também registramos os intermediários do modo reverso.

In [ ]:
shape_table = {
    "X": cache_small["X"].shape,
    "Z1": cache_small["Z1"].shape,
    "A1": cache_small["A1"].shape,
    "Z2": cache_small["Z2"].shape,
    "G2": aux_small["G2"].shape,
    "dA1": aux_small["dA1"].shape,
    "G1": aux_small["G1"].shape,
    "dX": dX_small.shape,
}
assert shape_table == {k: (2, 2) for k in shape_table}
assert all(grads_small[k].shape == params_small[k].shape for k in params_small)
for name, shape in shape_table.items():
    print(f"{name:>3}: {shape}")

## 4. Vetorizado = média dos exemplos

Um exemplo isolado é um lote de tamanho 1. A média dos gradientes individuais deve reproduzir o gradiente médio do lote completo. Para (dX), cada linha individual deve ser dividida por (m) ao compará-la com o lote.

In [ ]:
def grads_by_example(X, Y, params):
    individual_grads, individual_dX = [], []
    for i in range(X.shape[0]):
        _, _, cache_i = forward(X[i:i+1], Y[i:i+1], params)
        grads_i, dX_i, _ = backward(cache_i)
        individual_grads.append(grads_i)
        individual_dX.append(dX_i[0])
    mean_grads = {
        name: np.mean([g[name] for g in individual_grads], axis=0)
        for name in params
    }
    return mean_grads, np.vstack(individual_dX)


loop_grads, loop_dX_unscaled = grads_by_example(X_small, Y_small, params_small)
errors = {name: np.max(np.abs(grads_small[name] - loop_grads[name])) for name in params_small}
dX_error = np.max(np.abs(dX_small - loop_dX_unscaled / X_small.shape[0]))
assert max(errors.values()) < 1e-14
assert dX_error < 1e-14
print("Erros máximos, vetorizado × média individual:", errors)
print(f"Erro máximo em dX: {dX_error:.3e}")

## 5. Invariância a permutação

Trocar a ordem das linhas não muda uma média. Logo, loss e gradientes de parâmetros permanecem iguais. (dX), que pertence a exemplos específicos, acompanha a permutação.

In [ ]:
perm = np.array([1, 0])
loss_perm, P_perm, cache_perm = forward(X_small[perm], Y_small[perm], params_small)
grads_perm, dX_perm, _ = backward(cache_perm)

assert np.allclose(loss_perm, loss_small, atol=1e-15)
assert np.allclose(P_perm, P_small[perm], atol=1e-15)
assert all(np.allclose(grads_perm[k], grads_small[k], atol=1e-15) for k in params_small)
assert np.allclose(dX_perm, dX_small[perm], atol=1e-15)
print(f"Diferença na loss: {abs(loss_perm - loss_small):.3e}")
print("Gradientes de parâmetros invariantes; dX equivarante à permutação.")

## 6. Micro-lotes desiguais

Se cada micro-lote retorna um gradiente médio, combine-os por (g=\sum_r m_r g_r/\sum_rm_r). A média simples dos micro-lotes dá peso excessivo ao menor grupo.

In [ ]:
X_micro = rng.normal(size=(10, 3))
labels_micro = rng.integers(0, 3, size=10)
Y_micro = np.eye(3)[labels_micro]
params_micro = {
    "W1": rng.normal(0, 0.25, size=(3, 4)),
    "b1": rng.normal(0, 0.05, size=4),
    "W2": rng.normal(0, 0.25, size=(4, 3)),
    "b2": rng.normal(0, 0.05, size=3),
}

_, _, cache_full = forward(X_micro, Y_micro, params_micro)
grads_full, _, _ = backward(cache_full)

micro = []
for sl in (slice(0, 7), slice(7, 10)):
    _, _, cache_part = forward(X_micro[sl], Y_micro[sl], params_micro)
    g_part, _, _ = backward(cache_part)
    micro.append((X_micro[sl].shape[0], g_part))

weighted = {k: sum(n * g[k] for n, g in micro) / 10 for k in params_micro}
naive = {k: np.mean([g[k] for _, g in micro], axis=0) for k in params_micro}
weighted_error = max(np.max(np.abs(weighted[k] - grads_full[k])) for k in params_micro)
naive_error = max(np.max(np.abs(naive[k] - grads_full[k])) for k in params_micro)
assert weighted_error < 1e-14
assert naive_error > 1e-4
print(f"Erro ponderado × lote completo: {weighted_error:.3e}")
print(f"Erro da média ingênua: {naive_error:.6f}")

## 7. Verificação numérica curta

Para não antecipar a próxima aula, verificamos apenas algumas coordenadas por diferenças centrais. A Aula 14 tratará escolha de (\varepsilon), erro relativo e pontos não diferenciáveis de forma sistemática.

In [ ]:
def numerical_coordinate(X, Y, params, name, index, eps=1e-5):
    plus = {k: v.copy() for k, v in params.items()}
    minus = {k: v.copy() for k, v in params.items()}
    plus[name][index] += eps
    minus[name][index] -= eps
    loss_plus = forward(X, Y, plus)[0]
    loss_minus = forward(X, Y, minus)[0]
    return (loss_plus - loss_minus) / (2 * eps)


checks = [("W1", (0, 0)), ("W1", (1, 1)), ("b1", (0,)),
          ("W2", (0, 1)), ("W2", (1, 0)), ("b2", (1,))]
check_errors = []
for name, index in checks:
    numeric = numerical_coordinate(X_small, Y_small, params_small, name, index)
    analytic = grads_small[name][index]
    err = abs(numeric - analytic)
    check_errors.append(err)
    print(f"{name}{index}: analítico={analytic:+.9f}, numérico={numeric:+.9f}, erro={err:.3e}")

assert max(check_errors) < 1e-9
print(f"Erro máximo: {max(check_errors):.3e}")

## 8. Teste de descida local

Aplicamos um passo pequeno na direção negativa do gradiente. Isso é um teste de sinal e coerência, não um método completo de seleção de taxa de aprendizagem.

In [ ]:
def take_step(params, grads, learning_rate):
    return {name: params[name] - learning_rate * grads[name] for name in params}


step_params = take_step(params_small, grads_small, learning_rate=0.1)
loss_after = forward(X_small, Y_small, step_params)[0]
assert loss_after < loss_small
print(f"Loss antes:  {loss_small:.9f}")
print(f"Loss depois: {loss_after:.9f}")
print(f"Redução:     {loss_small - loss_after:.9f}")

## 9. Contraprova: média duplicada

Como (G_2) já contém (1/m), dividir `dW` e `db` outra vez reduz o gradiente por (m). O gráfico abaixo usa cópias idênticas do mesmo exemplo: o gradiente médio correto é invariável ao tamanho do lote; o incorreto decai como (1/m).

**Texto alternativo do gráfico:** linhas em escala logarítmica mostram a norma correta constante e a norma com média duplicada diminuindo à medida que o lote cresce.

In [ ]:
batch_sizes = np.array([1, 2, 4, 8, 16, 32])
correct_norms, double_mean_norms = [], []
for m in batch_sizes:
    X_rep = np.repeat(X_small[:1], m, axis=0)
    Y_rep = np.repeat(Y_small[:1], m, axis=0)
    _, _, cache_rep = forward(X_rep, Y_rep, params_small)
    grads_rep, _, _ = backward(cache_rep)
    norm = np.linalg.norm(grads_rep["W2"])
    correct_norms.append(norm)
    double_mean_norms.append(norm / m)

correct_norms = np.array(correct_norms)
double_mean_norms = np.array(double_mean_norms)
assert np.allclose(correct_norms, correct_norms[0], rtol=1e-14, atol=1e-14)
assert np.allclose(double_mean_norms, correct_norms[0] / batch_sizes)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(batch_sizes, correct_norms, "o-", label="redução correta")
ax.plot(batch_sizes, double_mean_norms, "s--", label="média duplicada")
ax.set(xlabel="tamanho do lote m", ylabel="norma de dW2",
       title="Uma segunda média altera a escala do gradiente")
ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
print(f"Norma correta: {correct_norms[0]:.9f}")
print(f"Razão incorreta/correta em m=32: {double_mean_norms[-1]/correct_norms[-1]:.6f}")

## 10. Cache imutável

O backward deve usar os valores do forward correspondente. Como o cache contém cópias, modificar o dicionário de parâmetros depois do forward não altera seus intermediários.

In [ ]:
params_mutable = {k: v.copy() for k, v in params_small.items()}
_, _, cache_before_mutation = forward(X_small, Y_small, params_mutable)
W1_cached = cache_before_mutation["W1"].copy()
params_mutable["W1"] += 100.0

assert np.array_equal(cache_before_mutation["W1"], W1_cached)
assert not np.array_equal(params_mutable["W1"], W1_cached)
grads_cached, _, _ = backward(cache_before_mutation)
assert np.allclose(grads_cached["W1"], grads_small["W1"])
print("O cache permaneceu consistente após a mutação externa dos parâmetros.")

## 11. Ensaio sintético de treinamento

Geramos três nuvens bidimensionais e treinamos somente para testar se o backward compõe vários passos coerentes. Todo o conjunto participa do ensaio; portanto, a acurácia abaixo é **de treinamento**, não uma estimativa de generalização nem resultado de seleção de modelo.

In [ ]:
centers = np.array([[-1.5, -1.0], [1.5, -0.8], [0.0, 1.5]])
labels = np.repeat(np.arange(3), 60)
X_train = centers[labels] + rng.normal(0, 0.45, size=(labels.size, 2))
Y_train = np.eye(3)[labels]
X_train = (X_train - X_train.mean(axis=0)) / X_train.std(axis=0)

train_params = {
    "W1": rng.normal(0, 0.25, size=(2, 8)),
    "b1": np.zeros(8),
    "W2": rng.normal(0, 0.25, size=(8, 3)),
    "b2": np.zeros(3),
}
losses = []
for step in range(400):
    loss, probabilities, cache = forward(X_train, Y_train, train_params)
    grads, _, _ = backward(cache)
    train_params = take_step(train_params, grads, learning_rate=0.15)
    losses.append(loss)

final_loss, final_probabilities, _ = forward(X_train, Y_train, train_params)
train_accuracy = np.mean(final_probabilities.argmax(axis=1) == labels)
assert final_loss < 0.08 * losses[0]
assert train_accuracy > 0.98
assert np.all(np.isfinite(losses))
print(f"Loss inicial: {losses[0]:.9f}")
print(f"Loss final:   {final_loss:.9f}")
print(f"Acurácia de treinamento: {train_accuracy:.6f}")

## 12. Auditoria final

Consolidamos os contratos mais importantes. Se esta célula falhar, o laboratório não deve ser publicado.

In [ ]:
audit = {
    "probabilidades_normalizadas": np.allclose(P_small.sum(axis=1), 1.0),
    "loss_finita": np.isfinite(loss_small),
    "gradientes_finitos": all(np.all(np.isfinite(g)) for g in grads_small.values()),
    "shapes_corretos": all(grads_small[k].shape == params_small[k].shape for k in params_small),
    "dx_shape_correto": dX_small.shape == X_small.shape,
    "vetorizado_igual_loop": max(errors.values()) < 1e-14,
    "permutacao_invariante": all(np.allclose(grads_perm[k], grads_small[k]) for k in params_small),
    "micro_lotes_ponderados": weighted_error < 1e-14,
    "media_ingenua_detectada": naive_error > 1e-4,
    "verificacao_numerica": max(check_errors) < 1e-9,
    "passo_reduz_loss": loss_after < loss_small,
    "media_duplicada_detectada": np.isclose(double_mean_norms[-1] / correct_norms[-1], 1/32),
    "cache_isolado": np.array_equal(cache_before_mutation["W1"], W1_cached),
    "ensaio_convergiu": final_loss < 0.08 * losses[0],
}
assert len(audit) == 14 and all(audit.values())
for name, passed in audit.items():
    print(f"[{'OK' if passed else 'FALHA'}] {name}")
print(f"\n{sum(audit.values())}/{len(audit)} grupos de auditoria aprovados.")

## Conclusões

- (A^\top G) acumula no peso as contribuições de todos os exemplos.
- O backward do bias desfaz o broadcasting com soma no eixo 0.
- O fator da redução média aparece exatamente uma vez.
- A versão vetorizada coincide com a média dos cálculos individuais.
- Micro-lotes desiguais exigem média ponderada por contagem.
- Shapes, permutação, descida local e diferenças centrais formam um conjunto de testes localizados.

Na próxima aula, a verificação numérica será transformada em um protocolo completo de **gradient checking**.